# 🧹 Tahap 2 — Data Cleaning & Processing

Tujuan: Bersihkan dan siapkan data agar siap digunakan untuk EDA dan Machine Learning.

Yang akan kita lakukan:
1. Load data mentah
2. Filter hanya pertandingan FIFA World Cup
3. Cek dan tangani missing values
4. Buat kolom target: hasil match (W/D/L)
5. Feature engineering
6. Simpan data bersih ke `data/processed/`

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print('Library siap ✅')

Library siap ✅


## 2. Load Data Mentah

In [2]:
df = pd.read_csv('../data/raw/results.csv')

print(f'Total baris  : {len(df):,}')
print(f'Total kolom  : {df.shape[1]}')
df.head()

Total baris  : 49,477
Total kolom  : 9


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## 3. Filter — Hanya FIFA World Cup

Dari ~49.000 pertandingan, kita hanya butuh yang **FIFA World Cup**.

In [3]:
df_wc = df[df['tournament'] == 'FIFA World Cup'].copy()
df_wc = df_wc.reset_index(drop=True)

print(f'Total pertandingan World Cup : {len(df_wc)}')
print(f'Tahun pertama                : {df_wc["date"].min()}')
print(f'Tahun terakhir               : {df_wc["date"].max()}')
df_wc.head()

Total pertandingan World Cup : 1036
Tahun pertama                : 1930-07-13
Tahun terakhir               : 2026-06-27


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True
2,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True
3,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
4,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True


## 4. Cek Missing Values

In [4]:
print('Missing values per kolom:')
print(df_wc.isnull().sum())
print()
print(f'Total baris dengan missing values: {df_wc.isnull().any(axis=1).sum()}')

Missing values per kolom:
date           0
home_team      0
away_team      0
home_score    56
away_score    56
tournament     0
city           0
country        0
neutral        0
dtype: int64

Total baris dengan missing values: 56


In [5]:
# Hapus baris jika ada missing values di kolom penting
kolom_penting = ['home_team', 'away_team', 'home_score', 'away_score']
df_wc = df_wc.dropna(subset=kolom_penting)

print(f'Jumlah baris setelah drop missing: {len(df_wc)}')

Jumlah baris setelah drop missing: 980


## 5. Konversi Tipe Data

In [6]:
# Konversi date ke datetime
df_wc['date'] = pd.to_datetime(df_wc['date'])

# Ekstrak tahun
df_wc['year'] = df_wc['date'].dt.year

# Pastikan skor bertipe integer
df_wc['home_score'] = df_wc['home_score'].astype(int)
df_wc['away_score'] = df_wc['away_score'].astype(int)

print('Tipe data sekarang:')
print(df_wc.dtypes)

Tipe data sekarang:
date          datetime64[ns]
home_team             object
away_team             object
home_score             int64
away_score             int64
tournament            object
city                  object
country               object
neutral                 bool
year                   int32
dtype: object


## 6. Buat Kolom Target: Hasil Match (W/D/L)

Target kita adalah hasil match dari **perspektif tim home**:
- **W** (Win)  = home_score > away_score
- **D** (Draw) = home_score == away_score  
- **L** (Loss) = home_score < away_score

In [7]:
def tentukan_hasil(row):
    if row['home_score'] > row['away_score']:
        return 'W'
    elif row['home_score'] == row['away_score']:
        return 'D'
    else:
        return 'L'

df_wc['result'] = df_wc.apply(tentukan_hasil, axis=1)

print('Distribusi hasil match:')
print(df_wc['result'].value_counts())
print()
print('Dalam persentase:')
print(df_wc['result'].value_counts(normalize=True).round(3) * 100)

Distribusi hasil match:
result
W    446
L    312
D    222
Name: count, dtype: int64

Dalam persentase:
result
W    45.5
L    31.8
D    22.7
Name: proportion, dtype: float64


## 7. Feature Engineering

Buat fitur-fitur yang akan digunakan model ML.

### 7a. Selisih Gol (Goal Difference)

In [8]:
df_wc['goal_diff'] = df_wc['home_score'] - df_wc['away_score']

print('Contoh data dengan goal_diff:')
df_wc[['home_team', 'away_team', 'home_score', 'away_score', 'goal_diff', 'result']].head(10)

Contoh data dengan goal_diff:


,home_team,away_team,home_score,away_score,goal_diff,result
0,Belgium,United States,0,3,-3,L
1,France,Mexico,4,1,3,W
2,Brazil,Yugoslavia,1,2,-1,L
3,Peru,Romania,1,3,-2,L
4,Argentina,France,1,0,1,W
5,Chile,Mexico,3,0,3,W
6,Bolivia,Yugoslavia,0,4,-4,L
7,Paraguay,United States,0,3,-3,L
8,Uruguay,Peru,1,0,1,W
9,Argentina,Mexico,6,3,3,W


### 7b. Win Rate Historis per Tim

Hitung win rate tiap tim berdasarkan **seluruh** data World Cup.
Ini menggambarkan seberapa kuat tim tersebut secara historis.

In [9]:
# Gabungkan data dari perspektif home dan away
home_records = df_wc[['home_team', 'result']].rename(columns={'home_team': 'team'})
home_records['win'] = (home_records['result'] == 'W').astype(int)

away_records = df_wc[['away_team', 'result']].rename(columns={'away_team': 'team'})
away_records['win'] = (away_records['result'] == 'L').astype(int)  # L untuk home = W untuk away

all_records = pd.concat([home_records, away_records])

# Hitung win rate per tim
win_rate = all_records.groupby('team')['win'].mean().reset_index()
win_rate.columns = ['team', 'win_rate']
win_rate['win_rate'] = win_rate['win_rate'].round(3)
win_rate = win_rate.sort_values('win_rate', ascending=False)

print('Top 10 tim dengan win rate tertinggi di World Cup:')
win_rate.head(10).reset_index(drop=True)

Top 10 tim dengan win rate tertinggi di World Cup:


,team,win_rate
0,Brazil,0.661
1,Germany,0.611
2,Italy,0.542
3,Netherlands,0.536
4,France,0.534
5,Argentina,0.534
6,Portugal,0.486
7,Hungary,0.469
8,Spain,0.456
9,Turkey,0.455


In [10]:
# Gabungkan win rate ke dataset utama
df_wc = df_wc.merge(win_rate.rename(columns={'team': 'home_team', 'win_rate': 'home_win_rate'}),
                    on='home_team', how='left')

df_wc = df_wc.merge(win_rate.rename(columns={'team': 'away_team', 'win_rate': 'away_win_rate'}),
                    on='away_team', how='left')

# Win rate ratio: seberapa jauh selisih kekuatan dua tim
df_wc['win_rate_diff'] = df_wc['home_win_rate'] - df_wc['away_win_rate']

print('Kolom baru berhasil ditambahkan:')
df_wc[['home_team', 'away_team', 'home_win_rate', 'away_win_rate', 'win_rate_diff', 'result']].head(10)

Kolom baru berhasil ditambahkan:


,home_team,away_team,home_win_rate,away_win_rate,win_rate_diff,result
0,Belgium,United States,0.404,0.263,0.141,L
1,France,Mexico,0.534,0.295,0.239,W
2,Brazil,Yugoslavia,0.661,0.424,0.237,L
3,Peru,Romania,0.278,0.381,-0.103,L
4,Argentina,France,0.534,0.534,0.000,W
5,Chile,Mexico,0.333,0.295,0.038,W
6,Bolivia,Yugoslavia,0.000,0.424,-0.424,L
7,Paraguay,United States,0.250,0.263,-0.013,L
8,Uruguay,Peru,0.417,0.278,0.139,W
9,Argentina,Mexico,0.534,0.295,0.239,W


### 7c. Apakah Lapangan Netral?

In [11]:
# Konversi kolom neutral ke integer (True=1, False=0)
df_wc['neutral'] = df_wc['neutral'].astype(int)

print('Distribusi lapangan netral vs kandang:')
print(df_wc['neutral'].value_counts())
print('0 = kandang salah satu tim, 1 = lapangan netral')

Distribusi lapangan netral vs kandang:
neutral
1    856
0    124
Name: count, dtype: int64
0 = kandang salah satu tim, 1 = lapangan netral


## 8. Pilih Kolom Final

In [12]:
kolom_final = [
    'date', 'year',
    'home_team', 'away_team',
    'home_score', 'away_score',
    'goal_diff',
    'home_win_rate', 'away_win_rate', 'win_rate_diff',
    'neutral',
    'result'
]

df_clean = df_wc[kolom_final].copy()

print(f'Shape data bersih: {df_clean.shape}')
print()
df_clean.head(10)

Shape data bersih: (980, 12)



,date,year,home_team,away_team,home_score,away_score,goal_diff,home_win_rate,away_win_rate,win_rate_diff,neutral,result
0,1930-07-13,1930,Belgium,United States,0,3,-3,0.404,0.263,0.141,1,L
1,1930-07-13,1930,France,Mexico,4,1,3,0.534,0.295,0.239,1,W
2,1930-07-14,1930,Brazil,Yugoslavia,1,2,-1,0.661,0.424,0.237,1,L
3,1930-07-14,1930,Peru,Romania,1,3,-2,0.278,0.381,-0.103,1,L
4,1930-07-15,1930,Argentina,France,1,0,1,0.534,0.534,0.000,1,W
5,1930-07-16,1930,Chile,Mexico,3,0,3,0.333,0.295,0.038,1,W
6,1930-07-17,1930,Bolivia,Yugoslavia,0,4,-4,0.000,0.424,-0.424,1,L
7,1930-07-17,1930,Paraguay,United States,0,3,-3,0.250,0.263,-0.013,1,L
8,1930-07-18,1930,Uruguay,Peru,1,0,1,0.417,0.278,0.139,0,W
9,1930-07-19,1930,Argentina,Mexico,6,3,3,0.534,0.295,0.239,1,W


## 9. Ringkasan Data Bersih

In [13]:
print('=== RINGKASAN DATA BERSIH ===')
print(f'Total pertandingan : {len(df_clean)}')
print(f'Rentang tahun      : {df_clean["year"].min()} - {df_clean["year"].max()}')
print(f'Missing values     : {df_clean.isnull().sum().sum()}')
print()
print('Distribusi target (result):')
print(df_clean['result'].value_counts())
print()
print('Statistik fitur numerik:')
df_clean[['home_win_rate', 'away_win_rate', 'win_rate_diff', 'neutral']].describe().round(3)

=== RINGKASAN DATA BERSIH ===
Total pertandingan : 980
Rentang tahun      : 1930 - 2026
Missing values     : 0

Distribusi target (result):
result
W    446
L    312
D    222
Name: count, dtype: int64

Statistik fitur numerik:


,home_win_rate,away_win_rate,win_rate_diff,neutral
count,980.000,980.000,980.000,980.000
mean,0.414,0.359,0.055,0.873
std,0.166,0.153,0.227,0.333
min,0.000,0.000,-0.661,0.000
25%,0.293,0.250,-0.095,1.000
50%,0.424,0.391,0.069,1.000
75%,0.536,0.455,0.223,1.000
max,0.661,0.661,0.661,1.000


## 10. Simpan Data Bersih

In [14]:
df_clean.to_csv('../data/processed/wc_clean.csv', index=False)
win_rate.to_csv('../data/processed/win_rate_per_team.csv', index=False)

print('Data berhasil disimpan!')
print('  → data/processed/wc_clean.csv')
print('  → data/processed/win_rate_per_team.csv')

Data berhasil disimpan!
  → data/processed/wc_clean.csv
  → data/processed/win_rate_per_team.csv


## ✅ Ringkasan Tahap 2

Yang sudah dilakukan:
- Filter 49k+ data → hanya World Cup matches
- Buat kolom target `result` (W/D/L) dari perspektif tim home
- Feature engineering: `goal_diff`, `home_win_rate`, `away_win_rate`, `win_rate_diff`
- Data bersih disimpan ke `data/processed/wc_clean.csv`

➡️ Lanjut ke **Notebook 03 — EDA**